In [13]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# -------- USER INPUT --------
emory_file = "/hpc/home/yy450/link_kamaleswaranlab/EmoryDataset/EMR_RAW/2022/CJSEPSIS_CPT_2022.dsv"
mimic_file = "/hpc/home/yy450/link_kamaleswaranlab/mimic_iv/mimic_flat_files/CPT_PROCEDURES.csv"
# ----------------------------

# Load DataFrames
df_emory = pd.read_csv(emory_file, sep='|')
df_mimic = pd.read_csv(mimic_file)

print(f"✅ Loaded Emory: {df_emory.shape}, MIMIC: {df_mimic.shape}")
print(f"📋 Emory Columns: {list(df_emory.columns)}")
print(f"📋 MIMIC Columns: {list(df_mimic.columns)}")

# 🧪 Filter Emory by medication_id
# if "medication_id" not in df_emory.columns:
#     raise ValueError("'medication_id' column not found in Emory file.")

✅ Loaded Emory: (1093405, 9), MIMIC: (186074, 5)
📋 Emory Columns: ['pat_id', 'csn', 'procedure_cpt_cd', 'procedure_cpt_desc', 'modifier_cpt_cd', 'procedure_dttm', 'procedure_day', 'modifier_cpt_seq_num', 'group_modifier_cpt_desc']
📋 MIMIC Columns: ['pat_id', 'csn', 'procedure_cpt_code', 'procedure_cpt_desc', 'procedure_dttm']


In [ ]:

# Emory analysis: list of medication IDs to filter by
# emory_medication_ids = [2208305]  # <-- Can add multiple ids e.g. [2196639, 2196640, 2196641]
emory_col = "procedure_cpt_cd"

# MIMIC analysis: directly analyze this column
mimic_col = "procedure_cpt_code"

In [ ]:


# df_emory_filtered = df_emory[df_emory["medication_id"].isin(emory_medication_ids)]
# print(f"🔍 Emory rows after filtering by medication_id {emory_medication_ids}: {df_emory_filtered.shape[0]} rows")

# Check column existence
if emory_col not in df_emory.columns:
    raise ValueError(f"'{emory_col}' not found in Emory file.")
if mimic_col not in df_mimic.columns:
    raise ValueError(f"'{mimic_col}' not found in MIMIC file.")


e_col = df_emory[emory_col]
# Extract target columns
# Transform to numeric, coercing errors to NaN
# e_col = pd.to_numeric(df_emory[emory_col], errors='coerce')

# Find invalid (non-numeric) entries
invalid_values = df_emory.loc[e_col.isna(), emory_col].unique()
print(f"🚨 None numeric type examples ({len(invalid_values)}):\n", invalid_values[:20])


m_col = df_mimic[mimic_col]

🚨 None numeric type examples (1):
 [nan]


In [18]:

# -- 1. Null Count --
print("\n🧼 Null Counts:")
print(f"Emory - {emory_col}: {e_col.isnull().sum()} nulls out of {len(e_col)}")
print(f"MIMIC - {mimic_col}: {m_col.isnull().sum()} nulls out of {len(m_col)}")



🧼 Null Counts:
Emory - procedure_cpt_cd: 6550 nulls out of 1093405
MIMIC - procedure_cpt_code: 0 nulls out of 186074


In [19]:
pd.options.display.float_format = '{:.4f}'.format

# -- 2. Summary Stats --
print("\n📊 Summary Statistics:")
print("\nEmory:")
display(e_col.describe(include='all'))
print("\nMIMIC:")
display(m_col.describe(include='all'))



📊 Summary Statistics:

Emory:


count     1086855
unique       3691
top         99233
freq       165737
Name: procedure_cpt_cd, dtype: object


MIMIC:


count     186074
unique      2366
top        G0378
freq       68571
Name: procedure_cpt_code, dtype: object

In [ ]:
def plot_column_dist(col, source_name, ax, value_range=None):
    if pd.api.types.is_numeric_dtype(col):
        col_to_plot = col.dropna()
        if value_range:
            col_to_plot = col_to_plot[(col_to_plot >= value_range[0]) & (col_to_plot <= value_range[1])]
        sns.histplot(col_to_plot, kde=True, bins=50, ax=ax)
        ax.set_title(f"{source_name} - Histogram")
        ax.set_xlabel(col.name)
    else:
        col.value_counts().plot(kind='bar', ax=ax)
        ax.set_title(f"{source_name} - Bar Chart")
        ax.set_ylabel("Count")
        ax.set_xlabel(col.name)

value_range = (0, 25)  # <- You can adjust this range based on expected values
# value_range = None  # <- No range filtering

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
plot_column_dist(e_col, "Emory", axes[0], value_range=value_range)
plot_column_dist(m_col, "MIMIC", axes[1], value_range=value_range)
plt.tight_layout()
plt.show()


In [20]:

# -- 3.5 Unique Values (for non-numeric) --
if not pd.api.types.is_numeric_dtype(e_col):
    print(f"\n📋 Emory unique values in '{emory_col}':")
    print(e_col.value_counts(dropna=False))

if not pd.api.types.is_numeric_dtype(m_col):
    print(f"\n📋 MIMIC unique values in '{mimic_col}':")
    print(m_col.value_counts(dropna=False))



📋 Emory unique values in 'procedure_cpt_cd':
procedure_cpt_cd
99233    165737
71045     73019
99232     71885
93010     68739
99291     52418
          ...  
63265         1
01742         1
29894         1
41006         1
51575         1
Name: count, Length: 3692, dtype: int64

📋 MIMIC unique values in 'procedure_cpt_code':
procedure_cpt_code
G0378    68571
99219    52408
99218    12091
99220    11069
44970     1174
         ...  
31002        1
S2900        1
42975        1
25635        1
23073        1
Name: count, Length: 2366, dtype: int64


In [ ]:
if pd.api.types.is_numeric_dtype(e_col) and pd.api.types.is_numeric_dtype(m_col):
    plt.figure(figsize=(10, 5))

    e_kde = e_col.dropna()
    m_kde = m_col.dropna()

    if value_range:
        e_kde = e_kde[(e_kde >= value_range[0]) & (e_kde <= value_range[1])]
        m_kde = m_kde[(m_kde >= value_range[0]) & (m_kde <= value_range[1])]

    sns.kdeplot(e_kde, label='Emory', fill=True)
    sns.kdeplot(m_kde, label='MIMIC', fill=True)
    plt.title(f"Emory ({emory_col}) vs MIMIC ({mimic_col}) - KDE Overlay")
    plt.xlabel("Value")
    plt.legend()
    plt.show()
else:
    print("\n⚠️ Skipping side-by-side KDE plot: one or both columns are not numeric.")



⚠️ Skipping side-by-side KDE plot: one or both columns are not numeric.
